https://grok.com/share/bGVnYWN5_252361af-3e16-420e-b901-14170e6d2eb0

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from scipy.spatial import distance

# Load the processed dataset
df = pd.read_csv('../Data/Processed/compas-scores-two-years-processed.csv')

# Drop all rows with null values
df = df.dropna()

# List of categorical features to label encode
label_features = [
    'sex', 'age_cat', 'race', 'c_charge_degree', 'r_charge_degree', 
    'vr_charge_degree', 'c_charge_desc', 'r_charge_desc', 'vr_charge_desc'
]

# Dictionary to store the LabelEncoders
label_encoders = {col: LabelEncoder() for col in label_features}

# Apply Label Encoding
for col in label_features:
    df[col] = label_encoders[col].fit_transform(df[col])

# Define target and features
y = df['two_year_recid']
X = df.drop(columns='two_year_recid')

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Load the trained Random Forest model
model = joblib.load('../Trained Models/rf_model.pkl')

# Function to predict probabilities
def predict_proba_rf(model, X):
    return model.predict_proba(X)[:, 1]  # Probability of recidivism (class 1)

# Define non-sensitive features for similarity metric
non_sensitive_features = ['priors_count', 'age_cat', 'c_charge_degree']

# Compute predictions for the test set
y_pred_proba = predict_proba_rf(model, X_test)

# Function to assess individual fairness
def assess_individual_fairness(X, y_pred, non_sensitive_features, threshold=0.1):
    # Extract non-sensitive features
    X_non_sensitive = X[non_sensitive_features].values
    
    # Initialize lists to store fairness violations
    fairness_violations = []
    similar_pairs = 0
    total_pairs = 0
    
    # Compare each pair of instances
    for i in range(len(X)):
        for j in range(i + 1, len(X)):
            # Compute Euclidean distance for non-sensitive features
            dist = distance.euclidean(X_non_sensitive[i], X_non_sensitive[j])
            
            # If individuals are "similar" (distance below a threshold)
            if dist < threshold:
                similar_pairs += 1
                pred_diff = abs(y_pred[i] - y_pred[j])
                if pred_diff > 0.1:  # Define a tolerance for prediction difference
                    fairness_violations.append((i, j, dist, pred_diff))
            total_pairs += 1
    
    # Calculate fairness violation rate
    violation_rate = len(fairness_violations) / similar_pairs if similar_pairs > 0 else 0
    
    return fairness_violations, violation_rate, similar_pairs, total_pairs

# Assess individual fairness on the test set
fairness_violations, violation_rate, similar_pairs, total_pairs = assess_individual_fairness(
    X_test, y_pred_proba, non_sensitive_features, threshold=0.5
)

# Print results
print(f"Number of similar pairs considered: {similar_pairs}")
print(f"Total pairs compared: {total_pairs}")
print(f"Individual fairness violation rate: {violation_rate:.2%}")
print(f"Number of fairness violations: {len(fairness_violations)}")
if fairness_violations:
    print("\nExamples of fairness violations (index i, index j, distance, prediction difference):")
    for i, j, dist, pred_diff in fairness_violations[:5]:  # Show top 5 violations
        print(f"Pair ({i}, {j}): Distance = {dist:.3f}, Pred Diff = {pred_diff:.3f}")

# Visualize fairness violations
if fairness_violations:
    violations_dist = [v[2] for v in fairness_violations]
    violations_pred_diff = [v[3] for v in fairness_violations]
    
    plt.figure(figsize=(10, 6))
    plt.scatter(violations_dist, violations_pred_diff, alpha=0.5)
    plt.xlabel('Similarity Distance (Non-Sensitive Features)')
    plt.ylabel('Prediction Difference')
    plt.title('Individual Fairness Violations')
    plt.axhline(y=0.1, color='r', linestyle='--', label='Fairness Threshold')
    plt.legend()
    plt.show()

# --- Optional: SHAP Explanation for Additional Insight ---
print("Generating SHAP Explanations...")
shap_explainer = shap.TreeExplainer(model)
shap_values = shap_explainer.shap_values(X_test)

# SHAP Summary Plot
plt.figure()
shap.summary_plot(shap_values, X_test, plot_type="bar")
plt.title("SHAP Summary Plot (Bar)")
plt.tight_layout()
plt.show()

# SHAP Dependence Plot for 'priors_count'
plt.figure()
shap.dependence_plot('priors_count', shap_values, X_test, interaction_index=None)
plt.title("SHAP Dependence Plot for 'priors_count'")
plt.tight_layout()
plt.show()